# 07 Attention Mechanisms & Scaled Dot-Product Attention

**Scenario**: 3-Token Scaled Dot-Product Attention Matrix Computation.

This notebook demonstrates step-by-step manual Scaled Dot-Product Attention computation ($Q, K, V \rightarrow QK^\top / \sqrt{d_k} \rightarrow \text{Softmax} \rightarrow \alpha V$) and verifies results against PyTorch `F.scaled_dot_product_attention`.

## Step 1: Step-by-Step Manual 3-Token Attention Computation

In [1]:
import numpy as np

# Given Input Matrices (T=3 tokens, dk=2)
Q = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
K = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0]])
V = np.array([[2.0, 1.0], [0.0, 2.0], [1.0, 0.0]])

dk = Q.shape[1]  # 2
scale = np.sqrt(dk)

# Step 1: QK^T
raw_scores = np.dot(Q, K.T)

# Step 2: Scaling by 1 / sqrt(dk)
scaled_scores = raw_scores / scale

# Step 3: Row-wise Softmax
exp_scores = np.exp(scaled_scores)
attn_weights = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

# Step 4: Weighted Sum over Values V
output = np.dot(attn_weights, V)

print("=== Manual Scaled Dot-Product Attention Verification ===")
print("Raw QK^T Scores:\n", raw_scores)
print("Scaled Scores (QK^T / sqrt(2)):\n", scaled_scores.round(4))
print("Softmax Attention Weights alpha:\n", attn_weights.round(4))
print("Attention Output Matrix:\n", output.round(4))

=== Manual Scaled Dot-Product Attention Verification ===
Raw QK^T Scores:
 [[1. 1. 0.]
 [0. 1. 1.]
 [1. 2. 1.]]
Scaled Scores (QK^T / sqrt(2)):
 [[0.7071 0.7071 0.    ]
 [0.     0.7071 0.7071]
 [0.7071 1.4142 0.7071]]
Softmax Attention Weights alpha:
 [[0.4011 0.4011 0.1978]
 [0.1978 0.4011 0.4011]
 [0.2483 0.5035 0.2483]]
Attention Output Matrix:
 [[1.     1.2033]
 [0.7967 1.    ]
 [0.7448 1.2552]]


## Step 2: Verification via PyTorch Scaled Dot-Product Attention

In [2]:
import torch
import torch.nn.functional as F

Q_tensor = torch.tensor(Q, dtype=torch.float32).unsqueeze(0)
K_tensor = torch.tensor(K, dtype=torch.float32).unsqueeze(0)
V_tensor = torch.tensor(V, dtype=torch.float32).unsqueeze(0)

torch_output = F.scaled_dot_product_attention(Q_tensor, K_tensor, V_tensor).squeeze(0).numpy()

print("=== PyTorch Native Attention Verification ===")
print("PyTorch Output Matrix:\n", torch_output.round(4))
print("Outputs Match Manual Calculation:", np.allclose(output, torch_output))

=== PyTorch Native Attention Verification ===
PyTorch Output Matrix:
 [[1.     1.2033]
 [0.7967 1.    ]
 [0.7448 1.2552]]
Outputs Match Manual Calculation: True


## Output Explanation & Complexity Analysis

- **$\sqrt{d_k}$ Scaling**: Scaling by $\sqrt{2} \approx 1.4142$ rescales variance to $1.0$, preventing Softmax saturation.
- **Attention Output Matrix**: Both manual calculation and PyTorch output identical matrix $[[1.0000, 1.2033], [0.7967, 1.0000], [0.7448, 1.2552]]$.
- **PyTorch Verification**: PyTorch `F.scaled_dot_product_attention` matches manual matrix computation 100% (`np.allclose = True`).
- **Time Complexity**: $O(T^2 \cdot d)$ per sequence.
- **Space Complexity**: $O(T^2)$ RAM for attention matrix storage.